# 🛡️ Transaction Fraud Scorer Agentic AI (Google Colab Setup & Run)

Welcome to the **Transaction Fraud Scorer Agentic AI** interactive Colab notebook!

This notebook demonstrates how to:
1. Verify hardware environment (GPU / CPU).
2. Clone repository & install dependencies (`langchain`, `langgraph`, `chromadb`, `fastapi`, `websockets`).
3. Start the FastAPI Gateway and expose it externally via a Cloudflare Quick Tunnel.
4. Install and launch the local **Ollama** server directly inside Google Colab.
5. Pull the **Heterogeneous Model Tiers** into Ollama (`llama3.2:3b`, `llama3.1:8b`).
6. Initialize the **ChromaDB Vector Database** and ingest Year-on-Year fraud policies (2025 vs 2026) for RAG.
7. Test SQLite MCP Risk Tools & ChromaDB Policy Retrieval Skill.
8. Initialize the **LangGraph Hybrid Supervisor-Critic** Orchestrator (`OrchestratorAgent`).
9. Evaluate Scenario 1: Normal local purchase (`APPROVE` with Evaluator `ACCEPTED`).
10. Evaluate Scenario 2: Multi-vector fraud purchase (`DECLINE` with RAG policy grounding).
11. Demonstrate Actor-Critic Evaluator audit/self-correction & evaluate custom transactions.
12. Verify API Gateway transaction scoring and real-time WebSocket event streaming.

## Step 1: Verify Hardware Environment
Check if a GPU is assigned. If running on CPU, `llama3.2:3b` (and `llama3.2:1b`) will still run smoothly!

In [1]:
!nvidia-smi || echo "Running in CPU mode. Lightweight models like llama3.2:3b run well on CPU too!"

Thu Nov 12 17:16:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Step 2: Clone Repository, Install Dependencies & Load Colab Secrets
Clone the repository, install Python dependencies, and securely import secrets (such as `LANGSMITH_API_KEY`, `GEMINI_API_KEY`) from Google Colab's 🔑 Secrets manager (`google.colab.userdata`).

In [ ]:
import os
import sys

REPO_URL = "https://github.com/daknamsubho/TransactionFraudScorerAgenticAI.git"
REPO_DIR = "/content/TransactionFraudScorerAgenticAI" if os.path.exists("/content") else os.path.abspath("TransactionFraudScorerAgenticAI")
BRANCH = "feature/local_models"

# Clone or update repository on the target branch
if not os.path.exists(os.path.join(REPO_DIR, ".git")):
    print(f"📥 Cloning {REPO_URL} (branch: {BRANCH})...")
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    print(f"🔄 Repository found at {REPO_DIR}. Fetching latest changes on {BRANCH}...")
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} checkout {BRANCH}
    !git -C {REPO_DIR} pull origin {BRANCH}

os.chdir(REPO_DIR)
print("📂 Working directory:", os.getcwd())

# Install project dependencies
!pip install -q "langchain>=1.4.0" "langchain-ollama>=1.1.0" "langgraph>=0.2.0" "mcp>=1.3.0"
!pip install -q "pydantic>=2.7.0" "pydantic-settings>=2.2.0" "rich>=13.7.0" "httpx>=0.27.0"
!pip install -q "chromadb>=0.5.0"
!pip install -q -e .

src_path = os.path.join(REPO_DIR, "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# Import secrets securely from Google Colab Secrets (the 🔑 Key icon on the left panel)
try:
    from google.colab import userdata

    print("\n🔑 Importing secrets from Google Colab userdata...")
    colab_secret_keys = [
        "LANGSMITH_API_KEY",
        "LANGCHAIN_API_KEY",
        "LANGSMITH_PROJECT",
        "LANGSMITH_ENDPOINT",
        "GEMINI_API_KEY",
        "OPENAI_API_KEY",
        "ANTHROPIC_API_KEY",
    ]
    imported_count = 0
    for key in colab_secret_keys:
        try:
            val = userdata.get(key)
            if val:
                os.environ[key] = str(val).strip()
                masked = str(val)[:6] + "..." if len(str(val)) > 6 else "***"
                print(f"  • {key}: Loaded ({masked})")
                imported_count += 1
        except Exception:
            pass

    # Automatically activate LangSmith tracing when an API key is provided
    if os.environ.get("LANGSMITH_API_KEY") or os.environ.get("LANGCHAIN_API_KEY"):
        os.environ["LANGSMITH_TRACING"] = "true"
        os.environ["LANGCHAIN_TRACING_V2"] = "true"
        if not os.environ.get("LANGSMITH_PROJECT"):
            os.environ["LANGSMITH_PROJECT"] = "fraud-scorer-agentic-ai"
        print(f"  • LangSmith Tracing: ENABLED (Project: {os.environ['LANGSMITH_PROJECT']})")
    elif imported_count == 0:
        print("  • No secrets found in Colab userdata. You can store LANGSMITH_API_KEY under the 🔑 Secrets tab.")
except (ImportError, ModuleNotFoundError):
    pass

print("✅ Environment and dependencies configured successfully!")


🔄 Repository found at /content/TransactionFraudScorerAgenticAI. Fetching latest changes on feature/local_models...
remote: Enumerating objects: 15, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 8 (delta 7), reused 8 (delta 7), pack-reused 0 (from 0)
Unpacking objects: 100% (8/8), 6.84 KiB | 636.00 KiB/s, done.
From https://github.com/daknamsubho/TransactionFraudScorerAgenticAI
   774663f..1342875  feature/local_models -> origin/feature/local_models
M	src/chromadb/chroma_data/chroma.sqlite3
M	src/chromadb/chroma_data/f54d244c-565b-45d4-ba54-9aeb4237affd/data_level0.bin
M	src/chromadb/chroma_data/f54d244c-565b-45d4-ba54-9aeb4237affd/length.bin
M	src/sqllite/fraud_scorer.db
Already on 'feature/local_models'
Your branch is behind 'origin/feature/local_models' by 1 commit, and can be fast-forwarded.
  (use "git pull" to update your local branch)
From https://github.com/daknamsubho/TransactionFraudScorerAgenticAI
 * branch   

INFO:     65.130.24.27:0 - "GET /health HTTP/1.1" 200 OK
INFO:     65.130.24.27:0 - "OPTIONS /api/v1/transactions/score HTTP/1.1" 200 OK
INFO:     65.130.24.27:0 - "POST /api/v1/transactions/score HTTP/1.1" 200 OK


## Step 3: Start FastAPI Gateway & Expose via Cloudflare Tunnel (`cloudflared`)
Start the FastAPI API Gateway (`src/api_gateway`) on port `8000` in the background and expose it externally via a **Cloudflare Quick Tunnel** (`https://*.trycloudflare.com`).

Copy the printed URLs into your local website client to ingest transactions and stream real-time WebSocket score events!

In [3]:
import threading
import time
import httpx
import uvicorn

from api_gateway.app import app
from api_gateway.tunnel import start_cloudflare_tunnel

GATEWAY_PORT = 8000

def _run_uvicorn() -> None:
    uvicorn.run(app, host="0.0.0.0", port=GATEWAY_PORT, log_level="info")

# Start FastAPI server in a daemon thread if not already listening
try:
    httpx.get(f"http://127.0.0.1:{GATEWAY_PORT}/health", timeout=1.0)
    print("✅ FastAPI Gateway is already running on port", GATEWAY_PORT)
except Exception:
    server_thread = threading.Thread(target=_run_uvicorn, daemon=True)
    server_thread.start()
    time.sleep(2)
    print("✅ FastAPI Gateway started on http://127.0.0.1:8000")

# Start Cloudflare Quick Tunnel
tunnel_proc, public_url = start_cloudflare_tunnel(port=GATEWAY_PORT)
ws_url = public_url.replace("https://", "wss://") + "/ws/transactions/events"

print("\n" + "=" * 72)
print("🌐 CLOUDFLARE TUNNEL ACTIVE — USE THESE ENDPOINTS IN YOUR LOCAL WEBSITE")
print("=" * 72)
print(f"🔗 Public Base URL : {public_url}")
print(f"📖 Swagger UI Docs : {public_url}/docs")
print(f"📨 Score Endpoint  : POST {public_url}/api/v1/transactions/score")
print(f"⚖️ Decision API    : POST {public_url}/api/v1/transactions/decision")
print(f"📡 WebSocket Stream: {ws_url}")
print("-" * 72)
print("🖥️  To launch the local WebUI on your computer, run:")
print("    uv run fraud-web-client --port 3000")
print(f"    Then open http://127.0.0.1:3000 and paste {public_url} into 'API Gateway Base URL'.")
print("=" * 72)

INFO:     Started server process [462]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


✅ FastAPI Gateway started on http://127.0.0.1:8000

🌐 CLOUDFLARE TUNNEL ACTIVE — USE THESE ENDPOINTS IN YOUR LOCAL WEBSITE
🔗 Public Base URL : https://eat-poems-pairs-travels.trycloudflare.com
📖 Swagger UI Docs : https://eat-poems-pairs-travels.trycloudflare.com/docs
📨 Score Endpoint  : POST https://eat-poems-pairs-travels.trycloudflare.com/api/v1/transactions/score
⚖️ Decision API    : POST https://eat-poems-pairs-travels.trycloudflare.com/api/v1/transactions/decision
📡 WebSocket Stream: wss://eat-poems-pairs-travels.trycloudflare.com/ws/transactions/events
------------------------------------------------------------------------
🖥️  To launch the local WebUI on your computer, run:
    uv run fraud-web-client --port 3000
    Then open http://127.0.0.1:3000 and paste https://eat-poems-pairs-travels.trycloudflare.com into 'API Gateway Base URL'.


## Step 4: Install Ollama in Colab & Launch Background Server
We install Ollama using its official one-line script and start the background daemon process on `http://localhost:11434`.

In [4]:
import subprocess
import time

import requests

print("📥 Updating and installing zstd...")
!sudo apt update
!sudo apt install -y pciutils
!sudo apt-get install zstd

print("📥 Downloading and installing Ollama...")
!curl -fsSL https://ollama.com/install.sh | sh

print("🚀 Starting Ollama daemon in background...")
process = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)

# Check health
for i in range(10):
    try:
        resp = requests.get("http://localhost:11434/api/version")
        if resp.status_code == 200:
            print(f"✅ Ollama server is UP and ready! Version: {resp.json().get('version')}")
            break
    except Exception:
        time.sleep(1)
else:
    print("⚠️ Ollama is still initializing. Please wait a moment.")

📥 Updating and installing zstd...
Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease [1,578 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:4 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]      
Get:5 https://cli.github.com/packages stable/main amd64 Packages [357 B]       
Hit:6 http://archive.ubuntu.com/ubuntu noble InRelease                         
Get:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  Packages [2,064 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]           
Get:9 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]        
Get:10 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ Packages [79.0 kB]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease [17.8 kB]m
Get:12 https://r2u.stat.illinois.edu/u

## Step 5: Pull Heterogeneous Open-Source Models (`llama3.2:3b`, `llama3.1:8b`)
Download the open-source model tiers used across the LangGraph multi-agent architecture:
- **Tier 1 (`llama3.2:3b`)**: Fast SLM for `DataRetrieverAgent` (SQLite & ChromaDB tool calling)
- **Tier 2 (`llama3.1:8b`)**: Analytical LLM for `AnalysisAgent` (multi-vector signal correlation)
- **Tier 3 (`llama3.1:8b`)**: Reasoning & Critical Auditor LLM for `DecisionAgent` (Actor) and `EvaluatorAgent` (Critic) (PRD-019)

In [15]:
print("📦 Pulling Tier 1 model (llama3.2:3b) for DataRetrieverAgent...")
!ollama pull llama3.2:3b

print("\n📦 Pulling Tier 2 & Tier 3 model (llama3.1:8b) for AnalysisAgent, DecisionAgent & EvaluatorAgent...")
!ollama pull llama3.1:8b

print("\n📋 Available local models:")
!ollama list

📦 Pulling Tier 1 model (llama3.2:3b) for DataRetrieverAgent...


📦 Pulling Tier 2 & Tier 3 model (llama3.1:8b) for AnalysisAgent, DecisionAgent & EvaluatorAgent...


📋 Available local models:
NAME           ID              SIZE      MODIFIED               
llama3.1:8b    46e0c10c039e    4.9 GB    Less than a second ago    
llama3.2:3b    a80c4f17acd5    2.0 GB    Less than a second ago    


## Step 6: Initialize ChromaDB Vector Database & Ingest Policy Documents (2025 vs 2026)
Set up the persistent local **ChromaDB** vector database and ingest the Year-on-Year fraud policy documents from `docs/policy-docs/` (`fraud-risk-policy-2025.md` and `fraud-risk-policy-2026.md`).
This provides semantic retrieval grounding for the RAG Knowledge Layer so that agents cite authoritative policy clauses and demonstrate how identical transactions flip dispositions across policy years.

In [6]:
# Run automated ChromaDB initialization & policy document ingestion
!bash src/chromadb/install_chromadb.sh

import sys
if "src/chromadb" not in sys.path:
    sys.path.insert(0, "src/chromadb")

from client import get_chroma_client, get_or_create_policy_collection
from query_policies import query_fraud_policies
from fastapi.testclient import TestClient
from api_gateway.app import create_app

client = get_chroma_client()
collection = get_or_create_policy_collection(client)
print(f"\n✅ ChromaDB active with {collection.count()} policy chunks ready for RAG grounding.")

# Live demonstration of API Gateway ChromaDB endpoints (/api/v1/vector/*)
gateway_client = TestClient(create_app())
stats_resp = gateway_client.get("/api/v1/vector/stats").json()
print(f"📊 API Gateway Vector Stats: {stats_resp['total_documents']} chunks | By Year: {stats_resp['chunks_per_year']}")

test_query = 'Cryptocurrency exchange purchase over $3000 MCC 6051'
print(f"\n🔍 API Gateway Semantic Search (POST /api/v1/vector/query): '{test_query}'")

q2025 = gateway_client.post("/api/v1/vector/query", json={"query": test_query, "top_k": 1, "policy_year": "2025"}).json()
print(f"📌 [2025 Grounding] Clause: {q2025['results'][0]['clause_id']} | {q2025['results'][0]['title']}")

q2026 = gateway_client.post("/api/v1/vector/query", json={"query": test_query, "top_k": 1, "policy_year": "2026"}).json()
print(f"📌 [2026 Grounding] Clause: {q2026['results'][0]['clause_id']} | {q2026['results'][0]['title']}")


🔧 Checking ChromaDB installation...
✅ ChromaDB package verified (version: 1.5.9)
🗄️  Initializing ChromaDB Vector Database at /content/TransactionFraudScorerAgenticAI/src/chromadb/chroma_data...
📦 ChromaDB version: 1.5.9
🗄️  Initialized ChromaDB at: /content/TransactionFraudScorerAgenticAI/src/chromadb/chroma_data
📥 Ingesting policy documents from: /content/TransactionFraudScorerAgenticAI/docs/policy-docs
/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100% 79.3M/79.3M [00:02<00:00, 33.4MiB/s]
✅ Ingested 26 policy chunks (2025: 13, 2026: 13)

🧪 Running YoY Verification Test: 'Cryptocurrency exchange purchase over $2500 MCC 6051'
  • 2025 Policy Match: POL-2025-MCC-6051 (Similarity: 44.86%)
  • 2026 Policy Match: POL-2026-MCC-6051 (Similarity: 45.00%)

🎉 ChromaDB Vector Database setup & YoY policy verification SUCCESSFUL!

✅ ChromaDB active with 26 policy chunks ready for RAG grounding.
📊 API Gateway Vector Stats: 26 chunks | By Year: {'2025': 13, '2026': 13}

🔍 API Gatewa

## Step 7: Test MCP Risk Tools & ChromaDB Policy Retrieval Skill
Verify that the SQLite MCP risk evaluation tools (Velocity, Geolocation, Merchant MCC, Account History) and the ChromaDB `PolicyRetrievalSkill` are properly loaded.

In [16]:
from fraud_scorer.agents.tools import get_langchain_fraud_tools
from fraud_scorer.skills.policy_retrieval_skill import query_fraud_policies_tool

tools = [*get_langchain_fraud_tools(), query_fraud_policies_tool]
print(f"Loaded {len(tools)} LangChain / LangGraph Agent Tools (SQLite + ChromaDB):")
for t in tools:
    summary = t.description.strip().splitlines()[0]
    print(f"  • {t.name}: {summary}")

# Run a test invocation of velocity check
test_velocity = tools[0].invoke({"account_id": "acc_normal_01", "window_minutes": 60})
print("\nSample SQLite Tool Invocation Output:", test_velocity)

Loaded 5 LangChain / LangGraph Agent Tools (SQLite + ChromaDB):
  • check_velocity_tool: Analyze transaction velocity, swipe frequency, and burst spend for an account.
  • check_geo_anomaly_tool: Detect impossible travel, extreme distances, and rapid geographic hops between transactions.
  • check_merchant_risk_tool: Evaluate Merchant Category Code (MCC) risk and commercial reputation.
  • lookup_account_history_tool: Retrieve historical account age, chargeback history, and trust profile.
  • query_fraud_policies_tool: Query ChromaDB vector database for authoritative fraud risk policy clauses.

Sample SQLite Tool Invocation Output: {'account_id': 'acc_normal_01', 'window_minutes': 60, 'data_source': 'mock_db', 'count_transactions': 1, 'sum_amount': 45.0, 'velocity_score': 10.0, 'is_burst': False, 'transaction_count': 1, 'cumulative_volume': 45.0, 'velocity_risk_score': 10.0, 'reasoning': 'Velocity within normal baseline: 1 transactions ($45.00).'}


## Step 8: Initialize LangGraph Hybrid Supervisor-Critic Orchestrator with Heterogeneous Model Tiering
Initialize `OrchestratorAgent` driving a compiled **LangGraph `StateGraph(FraudEvaluationState)`** with:
- **Tier 1 (`llama3.2:3b`)**: `DataRetrieverAgent` (concurrent SQLite + ChromaDB I/O via `asyncio.gather`)
- **Tier 2 (`llama3.1:8b`)**: `AnalysisAgent` (concurrent multi-vector skills + on-demand `request_supplementary_data`)
- **Tier 3 (`llama3.1:8b`)**: `DecisionAgent` (Actor) & `EvaluatorAgent` (Critic with up to 3 targeted retries and 1800ms circuit breaker, PRD-019)

In [18]:
from fraud_scorer.agents import (
    AgentModelConfig,
    EvaluatorAgent,
    FraudScorerOrchestrator,
    OrchestratorAgent,
    display_assessment,
    is_ollama_available,
)
from fraud_scorer.config import settings
from fraud_scorer.models.transaction import GeoLocation, MerchantInfo, PaymentMethod, Transaction

print("Ollama server connected:", is_ollama_available("http://localhost:11434"))

# Configure Heterogeneous Model Tiering (automatically falls back to llama3.2:3b if 8B not pulled locally)
model_tiers = AgentModelConfig(
    data_retrieval_model="llama3.2:3b",
    analysis_model="llama3.1:8b",
    decision_model="llama3.1:8b",
    evaluator_model="llama3.1:8b",
)

orchestrator = OrchestratorAgent(
    ollama_base_url="http://localhost:11434",
    provider="ollama",
    model_config=model_tiers,
    policy_year="2026",
)
print("\n✅ LangGraph Supervisor-Critic Orchestrator initialized!")
print(f"  • Tier 1 (DataRetrieverAgent): {orchestrator.model_config_tiers.data_retrieval_model}")
print(f"  • Tier 2 (AnalysisAgent):      {orchestrator.model_config_tiers.analysis_model}")
print(f"  • Tier 3 (DecisionAgent):      {orchestrator.model_config_tiers.decision_model}")
print(f"  • Tier 3+ (EvaluatorAgent):    {orchestrator.model_config_tiers.evaluator_model}")
langsmith_status = f"Enabled (Project: {settings.langsmith_project})" if settings.is_langsmith_enabled() else "Disabled (Offline Mode)"
print(f"  • LangSmith Tracing:          {langsmith_status}")


Ollama server connected: True

✅ LangGraph Supervisor-Critic Orchestrator initialized!
  • Tier 1 (DataRetrieverAgent): llama3.2:3b
  • Tier 2 (AnalysisAgent):      llama3.1:8b
  • Tier 3 (DecisionAgent):      llama3.1:8b
  • Tier 3+ (EvaluatorAgent):    llama3.1:8b


## Step 9: Scenario 1 - Normal Local Coffee Purchase (LangGraph State Execution)
Evaluate a legitimate transaction ($42.50 in San Francisco at Blue Bottle Coffee by an established account).
Expected outcome: **`APPROVE`** with `EvaluatorAgent` verdict **`ACCEPTED`**.

In [9]:
tx_normal = Transaction(
    account_id="acc_normal_01",
    amount=42.50,
    currency="USD",
    location=GeoLocation(latitude=37.7749, longitude=-122.4194, city="San Francisco", country="US"),
    payment_method=PaymentMethod(payment_type="credit_card", card_last_four="4242", is_card_present=True),
    merchant=MerchantInfo(merchant_id="merch_cafe_88", name="Blue Bottle Coffee", mcc="5499"),
)

print("🔍 Evaluating Scenario 1 through LangGraph StateGraph...")
state_1 = await orchestrator.run_workflow_state(tx_normal)
assessment_1 = await orchestrator.evaluate_transaction(tx_normal)
print(f"Evaluator Verdict: {state_1.evaluator_verdict} | Retries: {state_1.retry_count} | Fallback: {state_1.is_fallback}")
display_assessment(tx_normal, assessment_1)

🔍 Evaluating Scenario 1 through LangGraph StateGraph...
Evaluator Verdict: ACCEPTED | Retries: 0 | Fallback: False


╭───────────────────────────── Fraud Scorer Assessment (LangGraph Supervisor-Critic) ─────────────────────────────╮
│ Account ID: acc_normal_01                                                                                       │
│ Amount: $42.50 USD                                                                                              │
│ Merchant: Blue Bottle Coffee (MCC 5499)                                                                         │
│ Location: San Francisco (37.7749, -122.4194)                                                                    │
│ Overall Risk Score: 0.0 / 100.0                                                                                 │
│ Decision: APPROVE                                                                                               │
│ Tools Invoked: lookup_account_history_tool, query_fraud_policies_tool, check_merchant_risk_tool,                │
│ check_velocity_tool, check_geo_anomaly_tool                                                                     │
│ LangSmith Run ID: 950c7c3f-7b61-4f53-83c3-f48d8108e6dc                                                          │
│                                                                                                                 │
│ Transaction tx_362030bda0fa ($42.50 USD) aligns with account 'acc_normal_01' historical baseline, geographic    │
│ location, and low-risk merchant profile. Approved automatically. Grounded in policy clause(s):                  │
│ POL-2026-MCC-5411, POL-2026-MCC-5816. [LLM Decision Notes: Based on the account's established trust tier and    │
│ lack of elevated risk factors, the transaction is APPROVED.]                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                      Risk Factors for tx_362030bda0fa                       
┏━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Factor ┃ Category ┃ Severity ┃ Score (+) ┃ Evidence                       ┃
┡━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ None   │ -        │ LOW      │       0.0 │ No anomalous signals detected. │
└────────┴──────────┴──────────┴───────────┴────────────────────────────────┘

## Step 10: Scenario 2 - Multi-Vector Fraudulent Transaction with RAG Grounding
Evaluate a high-risk transaction ($3,500.00 in Tokyo at a casino shortly after London activity with high velocity and new untrusted account profile).
Expected outcome: **`DECLINE`** grounded in ChromaDB 2026 policy clauses and verified by `EvaluatorAgent`.

In [10]:
tx_suspicious = Transaction(
    account_id="acc_suspicious_01",
    amount=3500.00,
    currency="USD",
    location=GeoLocation(latitude=35.6762, longitude=139.6503, city="Tokyo", country="JP"),
    payment_method=PaymentMethod(payment_type="credit_card", card_last_four="9911", is_card_present=False),
    merchant=MerchantInfo(merchant_id="merch_casino_77", name="Tokyo High Roller Casino", mcc="7995"),
)

print("🔍 Evaluating Scenario 2 through LangGraph StateGraph...")
state_2 = await orchestrator.run_workflow_state(tx_suspicious)
assessment_2 = await orchestrator.evaluate_transaction(tx_suspicious)
print(f"Evaluator Verdict: {state_2.evaluator_verdict} | Retries: {state_2.retry_count} | Fallback: {state_2.is_fallback}")
display_assessment(tx_suspicious, assessment_2)

🔍 Evaluating Scenario 2 through LangGraph StateGraph...
Evaluator Verdict: ACCEPTED | Retries: 0 | Fallback: False


╭───────────────────────────── Fraud Scorer Assessment (LangGraph Supervisor-Critic) ─────────────────────────────╮
│ Account ID: acc_suspicious_01                                                                                   │
│ Amount: $3,500.00 USD                                                                                           │
│ Merchant: Tokyo High Roller Casino (MCC 7995)                                                                   │
│ Location: Tokyo (35.6762, 139.6503)                                                                             │
│ Overall Risk Score: 100.0 / 100.0                                                                               │
│ Decision: DECLINE                                                                                               │
│ Tools Invoked: lookup_account_history_tool, query_fraud_policies_tool, check_velocity_tool,                     │
│ check_geo_anomaly_tool, check_merchant_risk_tool                                                                │
│ LangSmith Run ID: 83abfd44-0dba-410e-b8c2-07a80aeb271c                                                          │
│                                                                                                                 │
│ Decision: DECLINE (Composite Score: 100.0/100). Triggered 5 risk factor(s): ACCOUNT_HISTORY, VELOCITY,          │
│ GEO_ANOMALY, MERCHANT_RISK, AMOUNT_ANOMALY. Grounded in policy clause(s): POL-2026-MCC-7995, POL-2026-MCC-6051. │
│ [LLM Decision Notes: The transaction is declined due to high risk account activity, unusual transaction amount, │
│ and merchant risk associated with online gambling, resulting in a composite score of 92.0.]                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                         Risk Factors for tx_7f02da42c6c9                                          
┏━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Factor          ┃ Category        ┃ Severity ┃ Score (+) ┃ Evidence                                             ┃
┡━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ ACCOUNT_HISTORY │ ACCOUNT_HISTORY │ HIGH     │      20.0 │ Account age is 4 days with 2 prior fraud disputes    │
│                 │                 │          │           │ (trust_tier='untrusted').                            │
│ VELOCITY        │ VELOCITY        │ HIGH     │      16.2 │ 0 transactions recorded in the last 60 minutes       │
│                 │                 │          │           │ (velocity_risk_score=65.0).                          │
│ GEO_ANOMALY     │ GEO_ANOMALY     │ MEDIUM   │      11.2 │ Unusual distance (9558.56 km) from previous location │
│                 │                 │          │           │ (prior location).                                    │
│ MERCHANT_RISK   │ MERCHANT_RISK   │ HIGH     │      17.5 │ Merchant MCC 7995 belongs to high-risk category:     │
│                 │                 │          │           │ Unknown Category.                                    │
│ AMOUNT_ANOMALY  │ AMOUNT_ANOMALY  │ HIGH     │      35.0 │ Transaction amount ($3,500.00) is 140.0x higher than │
│                 │                 │          │           │ historical account average ($25.00).                 │
└─────────────────┴─────────────────┴──────────┴───────────┴──────────────────────────────────────────────────────┘

## Step 11: Actor-Critic Evaluator Verification & Custom Transaction Test
Demonstrate how `EvaluatorAgent` detects policy/arithmetic contradictions on a flawed draft assessment, and evaluate a custom transaction.

In [11]:
# 1. Demonstrate EvaluatorAgent catching a policy contradiction (APPROVE on high score)
flawed_draft = assessment_2.model_copy(update={"decision": "APPROVE"})
critique = await orchestrator.evaluator_agent.evaluate_assessment(tx_suspicious, flawed_draft)
print("🛡️ Evaluator Critic Audit on Flawed Draft:")
print(f"  • Verdict: {critique.verdict} ({critique.rejection_category})")
print(f"  • Target Agent: {critique.target_agent}")
print(f"  • Remediation: {critique.remediation_instruction}\n")

# 2. Evaluate a custom transaction through the full LangGraph pipeline
custom_tx = Transaction(
    account_id="acc_normal_01",
    amount=2400.00,
    currency="USD",
    location=GeoLocation(latitude=40.7128, longitude=-74.0060, city="New York", country="US"),
    payment_method=PaymentMethod(payment_type="credit_card", card_last_four="1234", is_card_present=False),
    merchant=MerchantInfo(merchant_id="merch_luxury_goods", name="Fifth Avenue Electronics", mcc="5732"),
)

print("🔍 Evaluating Custom Transaction...")
custom_assessment = await orchestrator.evaluate_transaction(custom_tx)
display_assessment(custom_tx, custom_assessment)

🛡️ Evaluator Critic Audit on Flawed Draft:
  • Verdict: REJECTED (POLICY_CONTRADICTION)
  • Target Agent: decision
  • Remediation: Disposition 'APPROVE' violates score taxonomy for score 100.0. Expected disposition 'DECLINE' (APPROVE < 30, REVIEW 30-69.9, DECLINE >= 70).

🔍 Evaluating Custom Transaction...


╭───────────────────────────── Fraud Scorer Assessment (LangGraph Supervisor-Critic) ─────────────────────────────╮
│ Account ID: acc_normal_01                                                                                       │
│ Amount: $2,400.00 USD                                                                                           │
│ Merchant: Fifth Avenue Electronics (MCC 5732)                                                                   │
│ Location: New York (40.7128, -74.006)                                                                           │
│ Overall Risk Score: 46.2 / 100.0                                                                                │
│ Decision: REVIEW                                                                                                │
│ Tools Invoked: lookup_account_history_tool, query_fraud_policies_tool, check_merchant_risk_tool,                │
│ check_velocity_tool, check_geo_anomaly_tool                                                                     │
│ LangSmith Run ID: fb7c1f3c-d577-42f3-98f8-1b5d703f332e                                                          │
│                                                                                                                 │
│ Decision: REVIEW (Composite Score: 46.25/100). Triggered 2 risk factor(s): GEO_ANOMALY, AMOUNT_ANOMALY.         │
│ Grounded in policy clause(s): POL-2026-MCC-5411, POL-2026-MCC-5816. [LLM Decision Notes: The transaction is     │
│ declined due to a high risk score (70.0) resulting from a significant geographic anomaly and an unusually large │
│ transaction amount, exceeding the account's historical average by 21.8 times.]                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                         Risk Factors for tx_3ac31d2d4604                                          
┏━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Factor         ┃ Category       ┃ Severity ┃ Score (+) ┃ Evidence                                               ┃
┡━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ GEO_ANOMALY    │ GEO_ANOMALY    │ MEDIUM   │      11.2 │ Unusual distance (4129.09 km) from previous location   │
│                │                │          │           │ (prior location).                                      │
│ AMOUNT_ANOMALY │ AMOUNT_ANOMALY │ HIGH     │      35.0 │ Transaction amount ($2,400.00) is 21.8x higher than    │
│                │                │          │           │ historical account average ($110.00).                  │
└────────────────┴────────────────┴──────────┴───────────┴────────────────────────────────────────────────────────┘

INFO:     65.130.24.27:0 - "GET /health HTTP/1.1" 200 OK
INFO:     65.130.24.27:0 - "OPTIONS /api/v1/transactions/score HTTP/1.1" 200 OK


INFO:     65.130.24.27:0 - "WebSocket /ws/transactions/events" [accepted]
INFO:     connection open


INFO:     65.130.24.27:0 - "GET /health HTTP/1.1" 200 OK
INFO:     65.130.24.27:0 - "POST /api/v1/transactions/score HTTP/1.1" 200 OK
INFO:     65.130.24.27:0 - "OPTIONS /api/v1/transactions/decision HTTP/1.1" 200 OK
INFO:     65.130.24.27:0 - "POST /api/v1/transactions/decision HTTP/1.1" 200 OK
INFO:     65.130.24.27:0 - "POST /api/v1/transactions/score HTTP/1.1" 200 OK


## Step 12: Verify API Gateway Returns Score Request via WebSocket (`Score=100.0`, `Status=APPROVE`)
Test that:
1. Sending a `POST /api/v1/transactions/score` request immediately returns `score: 100.0` and `status: "APPROVE"`.
2. Connected WebSocket clients on `/ws/transactions/events` receive the `WebSocketTransactionEvent` containing `transaction_id`, `username`, `amount`, `score: 100.0`, and `score_details` (including the echoed `request_payload`).
3. `POST /api/v1/transactions/decision` records an `Accept`/`Reject` decision with a `Reason`.

In [ ]:
import asyncio
import json
import httpx
import websockets

local_ws_url = f"ws://127.0.0.1:{GATEWAY_PORT}/ws/transactions/events"
local_http_url = f"http://127.0.0.1:{GATEWAY_PORT}"

sample_card_payload = {
    "username": "jdoe_99",
    "datetime": "2026-10-31T18:09:11Z",
    "ip": "198.51.100.42",
    "device_id": "dev_macbook_pro_91a",
    "location": {
        "latitude": 37.7749,
        "longitude": -122.4194,
        "city": "San Francisco",
        "state": "CA",
        "country": "US",
        "postal_code": "94103"
    },
    "transaction_details": {
        "transaction_type": "CARD",
        "amount": 1450.00,
        "currency": "USD",
        "merchant": {
            "merchant_id": "merch_88412",
            "name": "ElectroWorld Online",
            "mcc": "5732",
            "country": "US"
        },
        "card_details": {
            "card_type": "CREDIT",
            "card_network": "VISA",
            "card_last_four": "4242",
            "is_card_present": False,
            "expiration_month": 8,
            "expiration_year": 2028,
            "cvv_verified": True,
            "avs_result": "Y"
        }
    }
}

async with websockets.connect(local_ws_url) as ws:
    async with httpx.AsyncClient(timeout=30.0) as client:
        # 1. Score the transaction (returns score=100.0, status=APPROVE)
        score_resp = await client.post(f"{local_http_url}/api/v1/transactions/score", json=sample_card_payload)
        score_data = score_resp.json()
        print("✅ POST /api/v1/transactions/score Response:")
        print(json.dumps(score_data, indent=2))

        # 2. Receive real-time WebSocket event
        ws_msg = json.loads(await asyncio.wait_for(ws.recv(), timeout=5.0))
        print("\n📡 WebSocket Event Received (/ws/transactions/events):")
        print(json.dumps(ws_msg, indent=2))

        # 3. Submit Accept/Reject decision
        decision_payload = {
            "transaction_id": score_data["transaction_id"],
            "accept": True,
            "reason": "Customer verified transaction via 2FA challenge."
        }
        decision_resp = await client.post(f"{local_http_url}/api/v1/transactions/decision", json=decision_payload)
        print("\n⚖️ POST /api/v1/transactions/decision Response:")
        print(json.dumps(decision_resp.json(), indent=2))

INFO:     127.0.0.1:42120 - "WebSocket /ws/transactions/events" [accepted]
INFO:     connection open


INFO:     127.0.0.1:42132 - "POST /api/v1/transactions/score HTTP/1.1" 200 OK
✅ POST /api/v1/transactions/score Response:
{
  "transaction_id": "tx_50bd819d7998",
  "assessment_id": "assess_3854242783a6",
  "score": 22.5,
  "status": "APPROVE",
  "reasoning_summary": "Decision: APPROVE (Composite Score: 22.5/100). Triggered 2 risk factor(s): ACCOUNT_HISTORY, VELOCITY. Grounded in policy clause(s): POL-2026-MCC-5411, POL-2026-MCC-5816. [LLM Decision Notes: **Disposition: DECLINE (Composite Score: 85.0)** **Rationale:** The transaction is declined because account 'jdoe_99' is a 0-day-old, untrusted-tier account (risk score 85.0) with only 2 lifetime transactions, and the $1,450.00 charge\u2014equal to the account's entire average transaction value\u2014triggers heightened scrutiny ]",
  "risk_factors": [
    {
      "category": "ACCOUNT_HISTORY",
      "severity": "HIGH",
      "contribution_score": 20.0,
      "description": "Account age is 0 days with 0 prior fraud disputes (trust_tier

INFO:     65.130.24.27:0 - "GET /health HTTP/1.1" 200 OK


INFO:     65.130.24.27:0 - "WebSocket /ws/transactions/events" [accepted]
INFO:     connection open


INFO:     65.130.24.27:0 - "GET /api/v1/db/user-profiles?page=1&page_size=10 HTTP/1.1" 200 OK
INFO:     65.130.24.27:0 - "GET /api/v1/db/user-profiles?page=1&page_size=10 HTTP/1.1" 200 OK
INFO:     65.130.24.27:0 - "GET /api/v1/db/user-profiles?page=1&page_size=10 HTTP/1.1" 200 OK
INFO:     65.130.24.27:0 - "GET /health HTTP/1.1" 200 OK
INFO:     65.130.24.27:0 - "GET /api/v1/db/user-profiles?page=1&page_size=10 HTTP/1.1" 200 OK
INFO:     65.130.24.27:0 - "GET /api/v1/db/transaction-scores?page=1&page_size=10 HTTP/1.1" 200 OK
INFO:     65.130.24.27:0 - "GET /api/v1/vector/stats HTTP/1.1" 200 OK
INFO:     65.130.24.27:0 - "OPTIONS /api/v1/vector/query HTTP/1.1" 200 OK
INFO:     65.130.24.27:0 - "POST /api/v1/vector/query HTTP/1.1" 200 OK
INFO:     65.130.24.27:0 - "POST /api/v1/vector/query HTTP/1.1" 200 OK


INFO:     65.130.24.27:0 - "WebSocket /ws/transactions/events" [accepted]
INFO:     connection open


INFO:     65.130.24.27:0 - "OPTIONS /api/v1/transactions/score HTTP/1.1" 200 OK
INFO:     65.130.24.27:0 - "POST /api/v1/transactions/score HTTP/1.1" 200 OK
INFO:     65.130.24.27:0 - "OPTIONS /api/v1/transactions/decision HTTP/1.1" 200 OK
INFO:     65.130.24.27:0 - "POST /api/v1/transactions/decision HTTP/1.1" 200 OK
INFO:     65.130.24.27:0 - "POST /api/v1/transactions/score HTTP/1.1" 200 OK
INFO:     65.130.24.27:0 - "POST /api/v1/transactions/decision HTTP/1.1" 200 OK
INFO:     65.130.24.27:0 - "GET /api/v1/db/transactions?page=1&page_size=10 HTTP/1.1" 200 OK
INFO:     65.130.24.27:0 - "GET /api/v1/db/transaction-scores?page=1&page_size=10 HTTP/1.1" 200 OK
INFO:     65.130.24.27:0 - "GET /api/v1/db/transaction-scores?page=1&page_size=10 HTTP/1.1" 200 OK
INFO:     65.130.24.27:0 - "GET /api/v1/vector/stats HTTP/1.1" 200 OK
INFO:     65.130.24.27:0 - "POST /api/v1/vector/query HTTP/1.1" 200 OK
INFO:     65.130.24.27:0 - "POST /api/v1/vector/query HTTP/1.1" 200 OK
